In [1]:
import pandas as pd

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
data_source_path = r"C:\Users\venuv\Documents\ai developer\SentimentAnalysis\archive\imbd_reviews.csv".replace("\\", "/")
print(data_source_path)

C:/Users/venuv/Documents/ai developer/SentimentAnalysis/archive/imbd_reviews.csv


In [3]:
df = pd.read_csv(data_source_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
#remove all the html tags from the review column 
import re
modified_reviews = df["review"].apply(lambda x: re.sub("<.*?>", "", x))

In [5]:
modified_reviews[modified_reviews.apply(lambda x: x.strip()=="")] #checking for empty spaces
modified_reviews[modified_reviews.isna()] #checking for nan values

Series([], Name: review, dtype: object)

In [6]:
#data transformation
modified_reviews = modified_reviews.str.lower()\
.str.replace("[^\w\s]+", "", regex=True)\
.str.replace(r"\s+", " ")\
.str.strip()\
.str.split()
modified_reviews

0        [one, of, the, other, reviewers, has, mentione...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, this, was, a, wonderful, way, to,...
3        [basically, theres, a, family, where, a, littl...
4        [petter, matteis, love, in, the, time, of, mon...
                               ...                        
49995    [i, thought, this, movie, did, a, down, right,...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, am, a, catholic, taught, in, parochial, el...
49998    [im, going, to, have, to, disagree, with, the,...
49999    [no, one, expects, the, star, trek, movies, to...
Name: review, Length: 50000, dtype: object

In [7]:
df["review"] = modified_reviews

In [8]:
#checking output column
df["sentiment"].unique()
#There are only 2 values that are expected so no need to do any transformation

array(['positive', 'negative'], dtype=object)

In [9]:
from collections import Counter
wordCounts = Counter()


In [10]:
for tokens in df["review"]:
    wordCounts.update(tokens)

In [11]:
vocab_size = 20000
commonWords = wordCounts.most_common(vocab_size)


In [12]:
vocab = {"<UNK>": 0}
index = 1
for token, _ in commonWords:
    if token not in vocab:
        vocab[token] = index 
        index+=1

In [13]:
maxLen = 0
for tokens in df["review"]:
    maxLen = max(len(tokens), maxLen)

In [14]:
maxLen

2450

In [15]:
modifiedvectors = []
for i, tokens in enumerate(df["review"]):
    vector = [vocab.get(token, 0) for token in tokens]
    if len(vector)<maxLen:
        vector = vector+[0]*(maxLen-len(vector))
    modifiedvectors.append(vector)


In [16]:
df["review"] = modifiedvectors

In [17]:
#creating embeedings
import random
vocabSize = len(vocab)
dimensions = 50
embeedings_matrix = [
    [random.uniform(-0.5, 0.5) for _ in range(dimensions)] for _ in range(vocabSize)
]


In [18]:
def normalize(vector):
    norm = sum(x*x for x in vector)**0.5
    if(norm==0):
        return vector
    return [x/norm for x in vector]

In [19]:
def vectorizeSentence(sentence):
    vector = []
    for wordIndex in sentence:
        if wordIndex!=0:
            vector.append(embeedings_matrix[wordIndex])
    result = [0]*dimensions
    for vec in vector:
        for ind in range(dimensions):
            result[ind]+=vec[ind]
    return result if len(vector)==0 else normalize([x/len(vector) for x in result])


In [20]:
resizedVectors = []
for review in df["review"]:
    resizedVectors.append(vectorizeSentence(review))

In [21]:
df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)

C:\Users\venuv\AppData\Local\Temp\ipykernel_28316\3780438693.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)


In [22]:
df.head()

,review,sentiment
0,"[27, 4, 1, 77, 1928, 42, 1057, 11, 100, 144, 3...",1
1,"[3, 382, 113, 357, 1, 1351, 2985, 6, 50, 17889...",1
2,"[9, 191, 10, 12, 3, 382, 96, 5, 1104, 58, 19, ...",1
3,"[683, 221, 3, 234, 112, 3, 113, 435, 3617, 118...",0
4,"[0, 0, 109, 7, 1, 58, 4, 289, 6, 3, 2140, 1368...",1


In [23]:
weights = [random.uniform(-0.5, 0.5) for _ in range(dimensions)]
bias = 0

In [24]:
import math
def sigmoid(x):
    return 1/(1+math.exp(-x))

def predict(sentence, bias):
    val = sum([w*x for w, x in zip(weights, sentence)])
    val+=bias
    return sigmoid(val)


In [25]:
lr = 0.001
for epoch in range(100):
    loss = 0
    count = 0
    correct=0
    for tokens, y_actual in zip(df["review"], df["sentiment"]):
        sentence = vectorizeSentence(tokens)
        y_pred = predict(sentence, bias)
        pred_label = 1 if y_pred>0.5 else 0
        if(pred_label==y_actual):
            correct+=1
        error = y_actual-y_pred 

        for i in range(len(weights)):
            weights[i]+=lr*error*sentence[i]
        bias+=lr*error 

        #compute embeedings
        for token in tokens:
            if token>0:
                for dim in range(dimensions):
                    embeedings_matrix[token][dim]+=lr*error*weights[dim]

        loss += -(y_actual * math.log(y_pred + 1e-9) + (1 - y_actual) * math.log(1 - y_pred + 1e-9))
        count+=1

    print(f"epoch: {epoch}, loss: {loss/count}, accuracy: {correct/count}")

epoch: 0, loss: 0.5401751095886445, accuracy: 0.73522
epoch: 1, loss: 0.4116322351409099, accuracy: 0.81998
epoch: 2, loss: 0.35985951491859874, accuracy: 0.84684
epoch: 3, loss: 0.32993835207187194, accuracy: 0.8623
epoch: 4, loss: 0.30940932244839875, accuracy: 0.87282


KeyboardInterrupt: 